In [4]:
from func import *

# Read data

In [5]:
train_df = pd.read_csv("processed5/train_data.csv")
test_df = pd.read_csv("processed5/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

# Elastic Net

In [6]:
from sklearn.linear_model import ElasticNet

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8627505332713833 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [7]:
model = ElasticNet(alpha=res["best_params"]["model__alpha"],
                   l1_ratio=res["best_params"]["model__l1_ratio"])

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

print(f"R^2(log Price): {elastic_net_result['R2(log)']:.4f}")
print(f"R^2: {elastic_net_result['R2']:.4f}")
print(f"MAPE (%): {elastic_net_result['MAPE']:.2f}")
print(f"MdAPE (%): {elastic_net_result['MdAPE']:.2f}")

R^2(log Price): 0.8631
R^2: 0.7519
MAPE (%): 0.17
MdAPE (%): 11.90


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

rf_param_dist = {
    "model__n_estimators": randint(300, 1000),
    "model__max_depth": [None] + list(range(10, 60, 10)),
    "model__max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],
    "model__min_samples_leaf": randint(1, 15),
    "model__min_samples_split": randint(2, 20)
}

X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


model = make_model_pipeline(model=RandomForestRegressor(
    n_jobs=-1,
    random_state=42),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)


rs = RandomizedSearchCV(
    model,
    param_distributions=rf_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(rs.best_params_)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=rs.best_params_["model__n_estimators"],
                                 max_depth=rs.best_params_["model__max_depth"],
                                 max_features=rs.best_params_["model__max_features"],
                                 min_samples_leaf=rs.best_params_["model__min_samples_leaf"],
                                 min_samples_split=rs.best_params_["model__min_samples_split"])

rf_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

print(f"R^2(log Price): {rf_result['R2(log)']:.4f}")
print(f"R^2: {rf_result['R2']:.4f}")
print(f"MAPE (%): {rf_result['MAPE']:.2f}")
print(f"MdAPE (%): {rf_result['MdAPE']:.2f}")

# XGB

In [ ]:
from xgboost import XGBRegressor
from scipy.stats import randint, uniform

xgb_param_dist = {
    "model__n_estimators": randint(400, 1200),
    "model__learning_rate": uniform(0.01, 0.2),
    "model__max_depth": randint(3, 10),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__min_child_weight": randint(1, 10),
    "model__reg_lambda": uniform(0, 5),
    "model__reg_alpha": uniform(0, 2)
}

model = make_model_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
))

xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

In [ ]:
xgb_model = XGBRegressor(n_estimators=xgb_rs.best_params_["model__n_estimators"],
                         learning_rate=xgb_rs.best_params_["model__learning_rate"],
                         max_depth=xgb_rs.best_params_["model__max_depth"],
                         subsample=xgb_rs.best_params_["model__subsample"],
                         colsample_bytree=xgb_rs.best_params_["model__colsample_bytree"],
                         min_child_weight=xgb_rs.best_params_["model__min_child_weight"],
                         reg_lambda=xgb_rs.best_params_["model__reg_lambda"],
                         reg_alpha=xgb_rs.best_params_["model__reg_alpha"])

xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)

print(f"R^2(log Price): {xgb_result['R2(log)']:.4f}")
print(f"R^2: {xgb_result['R2']:.4f}")
print(f"MAPE (%): {xgb_result['MAPE']:.2f}")
print(f"MdAPE (%): {xgb_result['MdAPE']:.2f}")